In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem

fs = HfFileSystem()
test_files = fs.glob("datasets/magistermilitum/Tridis/data/test-*.parquet")
print(test_files)

labels = pq.read_table(test_files, columns=["Corpus", "Language"],
filesystem=fs).to_pandas()
print(labels.value_counts())

In [ ]:
table = pq.read_table(
    test_files,
    filesystem=fs,
    filters=[("Corpus", "=", "MLH"), ("Language", "=", "la")],
    )
mlh = table.to_pandas()

print(len(mlh), "lines")
print(mlh.columns.tolist())
print(mlh["text"].head())

In [ ]:
from pathlib import Path
from io import BytesIO
from PIL import Image

out = Path("data/mlh")
out.mkdir(parents=True, exist_ok=True)

names = []
truth = []
for i, row in mlh.iterrows():
    name = f"mlh_{i:04d}"
    image = Image.open(BytesIO(row["image"]))
    image.save(out / f"{name}.png")
    names.append(name)
    truth.append(f"{name} {row['text']}")

(out / "img_list.txt").write_text("\n".join(names) + "\n")
(out / "ground_truth.txt").write_text("\n".join(truth) + "\n")
print("saved", len(names), "lines to", out)

In [ ]:
from IPython.display import display

for name, text in list(zip(names, mlh["text"]))[:3]:
    print(name, "→", text)
    display(Image.open(out / f"{name}.png"))